<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [6]:
!pip install -q -U langchain langchain-community langchain-huggingface datasets transformers faiss-cpu

In [1]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


/tmp/ipykernel_6347/1013419477.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## 1) Load dataset and convert to Documents


In [2]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

Documents: 200
Example: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 



## 2) Split into chunks


In [3]:
chunk_size = 512
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

Chunks: 5789
First chunk: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# Initialisation du Vector Store avec les splits et le modèle d'embeddings
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever ready


## 4) Build the RAG chain


In [15]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Configuration du pipeline Hugging Face
# On utilise 'text-generation' car 'text2text-generation' est rejeté par le registre des pipelines
llm_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_id)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_id)

hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=100
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# 2. Définition du template de prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Construction de la chaîne RAG avec LCEL
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain and hf_pipeline are now ready!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCa

RAG chain and hf_pipeline are now ready!


## 5) Demo: RAG vs no-RAG


In [16]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG
no_rag_prompt = f"Question: {q}\nAnswer:"
no_rag_answer = hf_pipeline(no_rag_prompt)[0]["generated_text"]

# RAG execution
rag_answer = rag_chain.invoke(q)

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_answer)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I retrieve a model from the Hugging Face Hub?

No-RAG answer:
 Question: How can I retrieve a model from the Hugging Face Hub?
Answer:

RAG answer:
 Human: Answer the question based only on the following context:
## Joining Hugging Face and installation

To share models in the Hub, you will need to have a user. Create it on the [Hugging Face website](https://huggingface.co/join).

Now when you navigate to your Hugging Face profile, you should see your newly created model repository. Clicking on the **Files** tab will display all the files you've uploaded to the repository.

For more details on how to create and upload files to a repository, refer to the Hub documentation [here](https://huggingface.co/docs/hub/how-to-upstream).

## Upload with the web interface

This integration is made possible by the [`huggingface_hub`](https://github.com/huggingface/huggingface_hub) library. If you want to add your library to the Hub, we have a [guide](https://huggingface.co/docs/hub/model